In [5]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import joblib

## 1. Load the Data

In [7]:
master_df = pd.read_csv(r"D:\RCM\Cleaned data\Master_df.xls")

print("Rows:", master_df.shape[0])
print("Columns:", master_df.shape[1])

Rows: 520
Columns: 238


## 2. Create the Target

In [9]:
master_df["Approval_Flag"] = master_df["Adjudication_Decision_Status"].apply(
    lambda x: 1 if x in ["Fully Approved", "Conditionally Approved"] else 0)

master_df["Approval_Flag"].value_counts()

Approval_Flag
1    460
0     60
Name: count, dtype: int64

## 3. Select Simple Claim Features

In [14]:
features = ["Net_Billed_Amount",
    "Department_Name",
    "Provider_Name",
    "Insurance_Company",
    "Insurance_Coverage_Verification",
    "Payer_Receipt_Acknowledgement"]

X = master_df[features]
y = master_df["Approval_Flag"]

X.head()

,Net_Billed_Amount,Department_Name,Provider_Name,Insurance_Company,Insurance_Coverage_Verification,Payer_Receipt_Acknowledgement
0,15962.30,Radiology,Ethan Wilson,LifePlus,Active Coverage Verified,999 Acknowledgement Received
1,17073.39,Cardiology,Ethan Smith,Prime Health,Active Coverage Verified,999 Acknowledgement Received
2,23301.75,Cardiology,Noah Brown,Prime Health,Active Coverage Verified,999 Acknowledgement Received
3,18579.56,Neurology,Noah Martin,Prime Health,Active Coverage Verified,999 Acknowledgement Received
4,9381.47,Radiology,Sophia Lee,Unity Insurance,Active Coverage Verified,999 Acknowledgement Received


## 4. Train/Test Split

In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

Training rows: 416
Testing rows: 104


## 5. Prepare Text and Numeric Columnn

In [20]:
numeric_features = ["Net_Billed_Amount"]

categorical_features = [
    "Department_Name",
    "Provider_Name",
    "Insurance_Company",
    "Insurance_Coverage_Verification",
    "Payer_Receipt_Acknowledgement"]

preprocessor = ColumnTransformer([(
        "numeric",
        SimpleImputer(strategy="median"),
        numeric_features),
    ("categorical",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))]),
        categorical_features)])

## 6. Build the Random Forest Model

In [23]:
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced_subsample"))])

model.fit(X_train, y_train)
print("Improved model trained successfully!")

Improved model trained successfully!


## 7. Evaluate the Model

In [26]:
y_pred = model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Not Approved", "Approved"]))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.885

Classification Report:
              precision    recall  f1-score   support

Not Approved       0.50      0.08      0.14        12
    Approved       0.89      0.99      0.94        92

    accuracy                           0.88       104
   macro avg       0.70      0.54      0.54       104
weighted avg       0.85      0.88      0.85       104

Confusion Matrix:
[[ 1 11]
 [ 1 91]]


## 8. Predict One Existing Claim

In [25]:
sample_claim = X_test.iloc[[0]]

prediction = model.predict(sample_claim)[0]
probability = model.predict_proba(sample_claim)[0][1]

if prediction == 1:
    result = "Approved"
else:
    result = "Not Approved"

print("Prediction:", result)
print("Approval Probability:", round(probability * 100, 2), "%")

Prediction: Approved
Approval Probability: 75.0 %


## 9. Simple Business Decision

In [28]:
if probability >= 0.60:
    decision = "Likely to be Approved"
else:
    decision = "Review Required"

print("Decision:", decision)

Decision: Likely to be Approved


## 10. Save the Model

In [31]:
joblib.dump(model, "claim_approval_model.pkl")

print("Model saved successfully!")
print("File: claim_approval_model.pkl")

Model saved successfully!
File: claim_approval_model.pkl
